# 👟 ARShoe M1v2 13-Keypoint Fresh Training & Evaluation in Google Colab

This notebook trains a fresh **ARShoe M1v2** model on the **13-keypoint dataset** (`dataset/merged_v3_v4_13kp`) with:
- 3 keypoints removed: `ball_top` (KP 5), `ankle_center` (KP 12), and `shin_mid` (KP 15)
- 16-limb PAF topology (32 channels)
- Production skeleton-consistent decoder with data-driven 13-KP priors
- Full validation evaluation & side-by-side comparison against the 16-KP baseline

In [ ]:
# Step 1: Check GPU environment
!nvidia-smi

In [ ]:
# Step 2: Install required dependencies
!pip install -q pyyaml tqdm albumentations opencv-python matplotlib

In [ ]:
# Step 3: Extract project archive (if uploaded as zip) or set working directory
import os
from pathlib import Path

# If you uploaded colab_13kp_training_pack.zip:
if os.path.exists('colab_13kp_training_pack.zip'):
    print('Extracting colab_13kp_training_pack.zip...')
    !unzip -q colab_13kp_training_pack.zip
    print('Extraction complete!')

# Verify directory contents
print('Working Directory:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# Step 4: Verify 13-KP Dataset and Configurations
!python shoes_vto/tools/verify_13kp_dataset.py --dataset_root dataset/merged_v3_v4_13kp

In [ ]:
# Step 5: Launch 100-Epoch Fresh 13-KP Training
# With A100 or T4 GPU and torch.compile(), 100 epochs typically takes ~10-15 minutes!
!python shoes_vto/train_m1v2_13kp.py \
    --dataset_root dataset/merged_v3_v4_13kp \
    --output_dir shoes_vto/outputs/m1v2_13kp_training \
    --epochs 100 \
    --batch_size 16 \
    --lr 0.0005

In [ ]:
# Step 6: Evaluate Best Checkpoint on Validation Split
!python shoes_vto/evaluate_m1.py \
    --checkpoint shoes_vto/outputs/m1v2_13kp_training/checkpoints/best.pth \
    --dataset_root dataset/merged_v3_v4_13kp \
    --split valid \
    --output_dir shoes_vto/outputs/m1v2_13kp_valid_eval \
    --num_vis 10

In [ ]:
# Step 7: Compare 13-KP Results Directly Against the 16-KP Baseline
!python shoes_vto/tools/compare_13kp_vs_16kp.py \
    --eval_13kp shoes_vto/outputs/m1v2_13kp_valid_eval/evaluation_summary.json \
    --eval_16kp shoes_vto/shoes_vto/outputs/m1v2_valid_eval/evaluation_summary.json

In [ ]:
# Step 8: Package Checkpoints & Evaluation Reports for Download
!zip -r m1v2_13kp_training_results.zip shoes_vto/outputs/m1v2_13kp_training/checkpoints/best.pth shoes_vto/outputs/m1v2_13kp_valid_eval/
print('✅ Results packaged to m1v2_13kp_training_results.zip!')